In [1]:
import sys
import os
import matplotlib.pyplot as plt

import pandas as pd
from tools import plot_cluster_std_dev_boxplot_static_all_merge,plot_cluster_std_dev_boxplot_single
sys.path.append(os.path.abspath("../"))
from util import (
    Extractor,
    rules_to_dataframe)


In [2]:
from tqdm import tqdm
csv_files = [
    '/home/mabon/research/Autoyara/YaraResults/clusterCSV/ssdeep/th50.csv',
    '/home/mabon/research/Autoyara/YaraResults/clusterCSV/ssdeep/th60.csv',
    '/home/mabon/research/Autoyara/YaraResults/clusterCSV/ssdeep/th70.csv',
    '/home/mabon/research/Autoyara/YaraResults/clusterCSV/ssdeep/th80.csv',
    '/home/mabon/research/Autoyara/YaraResults/clusterCSV/ssdeep/th90.csv'
]

yara_files = [
    '/home/mabon/research/Autoyara/YaraResults/ruleEval/retrainedBloomFilters/ssdeep/AutoPyara/Th50rules/merged_group_1.yar',
    '/home/mabon/research/Autoyara/YaraResults/ruleEval/retrainedBloomFilters/ssdeep/AutoPyara/Th60rules/merged_group_1.yar',
    '/home/mabon/research/Autoyara/YaraResults/ruleEval/retrainedBloomFilters/ssdeep/AutoPyara/Th70rules/merged_group_1.yar',
    '/home/mabon/research/Autoyara/YaraResults/ruleEval/retrainedBloomFilters/ssdeep/AutoPyara/Th80rules/merged_group_1.yar',
    '/home/mabon/research/Autoyara/YaraResults/ruleEval/retrainedBloomFilters/ssdeep/AutoPyara/Th90rules/merged_group_1.yar'

]

df_listSdhash = []
mr=[5,5,5,5,5]
for csv_file, yara_file,mr in tqdm(zip(csv_files, yara_files,mr)):
    print(mr)
    df = Extractor(csv_file,yara_file,mr=mr)
    df_listSdhash.append(df)


yara_files = [
    '/home/mabon/research/Autoyara/YaraResults/ruleEval/retrainedBloomFilters/ssdeep/Autoyara/Th50rules/merged_group_1.yar',
    '/home/mabon/research/Autoyara/YaraResults/ruleEval/retrainedBloomFilters/ssdeep/Autoyara/Th60rules/merged_group_1.yar',
    '/home/mabon/research/Autoyara/YaraResults/ruleEval/retrainedBloomFilters/ssdeep/Autoyara/Th70rules/merged_group_1.yar',
    '/home/mabon/research/Autoyara/YaraResults/ruleEval/retrainedBloomFilters/ssdeep/Autoyara/Th80rules/merged_group_1.yar',
    '/home/mabon/research/Autoyara/YaraResults/ruleEval/retrainedBloomFilters/ssdeep/Autoyara/Th90rules/merged_group_1.yar'

]
mr=[5,5,5,5,5]

df_listSdhashYARA = []
for csv_file, yara_file,mr in tqdm(zip(csv_files, yara_files,mr)):
    print(mr)
    df = Extractor(csv_file,yara_file,mr=mr)
    df_listSdhashYARA.append(df)

0it [00:00, ?it/s]

5
LOG:max_cluster_num  8807


1it [01:36, 96.10s/it]

5
LOG:max_cluster_num  8874


2it [03:02, 90.22s/it]

...

3it [04:33, 90.45s/it]

5
LOG:max_cluster_num  8761


4it [05:58, 88.35s/it]

5
LOG:max_cluster_num  8588


5it [07:21, 88.31s/it]


In [11]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from tqdm import tqdm
from matplotlib.ticker import FixedLocator, FixedFormatter

# Global font settings
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.weight'] = 600
plt.rcParams['font.size'] = 22

# Font dictionaries
TITLE_FONT = {'family': 'DejaVu Sans', 'weight': 600, 'size': 22}
AXIS_FONT = {'family': 'DejaVu Sans', 'weight': 900, 'size': 18.0}
TICK_FONT = {'fontsize': 22, 'fontweight': 'bold'}
INSET_AXIS_FONT = {'family': 'DejaVu Sans', 'weight': 900, 'size': 14.0}  # Smaller font for inset

def plot_inset_cumulative_tp_auc(
    dfs_pyara, dfs_yara, 
    drop_smallest_n=0, 
    output_file='inset_cumulative_tp_auc_plot.pdf'
):
    if not all(isinstance(df, pd.DataFrame) for df in dfs_pyara + dfs_yara):
        raise ValueError("All items in input lists must be pandas DataFrames")
    if len(dfs_pyara) != 5 or len(dfs_yara) != 5:
        raise ValueError("Expected exactly 5 DataFrames in each list")
    if not isinstance(drop_smallest_n, int) or drop_smallest_n < 0:
        raise ValueError("drop_smallest_n must be a non-negative integer")

    # Collect all unique cluster sizes
    all_sizes = set()
    for df in dfs_pyara + dfs_yara:
        df = df.replace('None', np.nan).apply(pd.to_numeric, errors='coerce')
        try:
            sizes = df.columns.astype(int).tolist()
            all_sizes.update(sizes)
        except ValueError:
            continue

    all_sizes = sorted(all_sizes)
    if drop_smallest_n > 0 and len(all_sizes) > drop_smallest_n:
        all_sizes = all_sizes[drop_smallest_n:]
    if not all_sizes:
        raise ValueError("No valid cluster sizes found")

    # Filter DataFrames to only those cluster sizes
    filtered_dfs_pyara = []
    filtered_dfs_yara = []
    for df in dfs_pyara:
        df = df.replace('None', np.nan).apply(pd.to_numeric, errors='coerce')
        valid_cols = [col for col in df.columns if int(col) in all_sizes]
        filtered_dfs_pyara.append(df[valid_cols])
    for df in dfs_yara:
        df = df.replace('None', np.nan).apply(pd.to_numeric, errors='coerce')
        valid_cols = [col for col in df.columns if int(col) in all_sizes]
        filtered_dfs_yara.append(df[valid_cols])

    # Define custom ticks for cluster sizes
    tick_sizes = [0, 25, 50, 100, 1000]
    tick_positions = []
    for tick in tick_sizes:
        if all_sizes:
            closest_idx = min(range(len(all_sizes)), key=lambda i: abs(all_sizes[i] - tick))
            tick_positions.append(all_sizes[closest_idx])

    # Merge data across thresholds by averaging medians
    merged_medians_pyara = []
    merged_medians_yara = []
    valid_positions = all_sizes

    for size in all_sizes:
        medians_pyara = []
        medians_yara = []
        for df_pyara, df_yara in zip(filtered_dfs_pyara, filtered_dfs_yara):
            if size in df_pyara.columns:
                data_pyara = df_pyara[size].dropna().astype(float) * 100
                if not data_pyara.empty:
                    medians_pyara.append(np.median(data_pyara))
            if size in df_yara.columns:
                data_yara = df_yara[size].dropna().astype(float) * 100
                if not data_yara.empty:
                    medians_yara.append(np.median(data_yara))
        # Average the medians across thresholds
        merged_medians_pyara.append(np.mean(medians_pyara) if medians_pyara else np.nan)
        merged_medians_yara.append(np.mean(medians_yara) if medians_yara else np.nan)

    # Filter out NaN values for plotting
    valid_medians_pyara = [m for m in merged_medians_pyara if not np.isnan(m)]
    valid_medians_yara = [m for m in merged_medians_yara if not np.isnan(m)]
    valid_pos = [all_sizes[i] for i, (mp, my) in enumerate(zip(merged_medians_pyara, merged_medians_yara)) if not np.isnan(mp) and not np.isnan(my)]

    if not valid_medians_pyara or not valid_medians_yara:
        raise ValueError("No valid data to plot after merging thresholds")

    # Cumulative averages for AUC calculation
    cumulative_avg_pyara = np.cumsum(valid_medians_pyara) / np.arange(1, len(valid_medians_pyara) + 1)
    cumulative_avg_yara = np.cumsum(valid_medians_yara) / np.arange(1, len(valid_medians_yara) + 1)

    # Calculate AUC up to each cluster size
    auc_pyara = []
    auc_yara = []
    auc_diff = []
    for k in range(1, len(valid_pos) + 1):
        auc_p = np.trapz(cumulative_avg_pyara[:k], valid_pos[:k])
        auc_y = np.trapz(cumulative_avg_yara[:k], valid_pos[:k])
        auc_pyara.append(auc_p)
        auc_yara.append(auc_y)
        auc_diff.append(auc_p - auc_y)

    # Create main figure and axis
    fig, ax_main = plt.subplots(figsize=(15, 6))

    # Main Plot: AUC and AUC Difference
    ax_main.plot(valid_pos, auc_pyara, marker='o', linestyle='-', linewidth=2, color='#1f77b4', label='AutoPYara AUC')
    ax_main.plot(valid_pos, auc_yara, marker='s', linestyle='-', linewidth=2, color='#ff7f0e', label='AutoYara AUC')
    ax_main.plot(valid_pos, auc_diff, marker='^', linestyle='-', linewidth=2, color='#2ca02c', label='AUC Difference')
    ax_main.set_xscale('log')  # Log scale for cluster sizes
    ax_main.set_xticks(tick_positions)
    ax_main.set_xticklabels([str(t) for t in tick_sizes], **TICK_FONT)
    ax_main.set_xlabel('Cluster Size (Number of Items)', **TICK_FONT)
    ax_main.set_ylabel('AUC', **TICK_FONT)
    # Set explicit y-ticks and labels for AUC plot
    y_ticks = [0, 200000, 400000, 600000, 800000]
    y_labels = ['0', '2*e^5', '4*e^5', '6*e^5', '8*e^5']
    ax_main.yaxis.set_major_locator(FixedLocator(y_ticks))
    ax_main.yaxis.set_major_formatter(FixedFormatter(y_labels))
    ax_main.legend(
        loc='upper center',
        bbox_to_anchor=(0.5, 1.02),
        ncol=3,
        prop=AXIS_FONT,
        frameon=False
    )
    ax_main.grid(True, linestyle='--', alpha=0.5)

    # Create inset plot in top-left corner, moved down by 1 mm (~0.00656 in figure coordinates)
    ax_inset = fig.add_axes([0.18, 0.440, 0.3, 0.3])  # [left, bottom, width, height]
    ax_inset.plot(valid_pos, cumulative_avg_pyara, marker='o', linestyle='-', linewidth=1.5, color='#1f77b4', label='AutoPYara')  # AutoPYara
    ax_inset.plot(valid_pos, cumulative_avg_yara, marker='s', linestyle='-', linewidth=1.5, color='#ff7f0e', label='AutoYara')  # AutoYara
    # Fill the area between the curves
    ax_inset.fill_between(valid_pos, cumulative_avg_pyara, cumulative_avg_yara, color='red', alpha=0.3, label='Difference')
    ax_inset.set_xscale('log')
    # Modified Y-axis limits to increase zoom (more space)
    ax_inset.set_ylim(50, 100)  # Expanded range to open more space
    ax_inset.set_xticks(tick_positions)
    ax_inset.set_xticklabels([str(t) for t in tick_sizes], fontsize=8)
    ax_inset.set_yticks([50,75, 100])  # Keep simple y-ticks for TP rates
    ax_inset.set_yticklabels(['50','75','100'], fontsize=8)
    ax_inset.set_xlabel('Cluster Size', **INSET_AXIS_FONT)
    ax_inset.set_ylabel('TP Rate (%)', **INSET_AXIS_FONT)
    ax_inset.grid(True, linestyle='--', alpha=0.3)
    # Add legend to inset
    ax_inset.legend(loc='lower left', prop={'size': 8}, frameon=False)

    # Save the figure
    fig.subplots_adjust(top=0.85)
    fig.savefig(output_file, format='pdf', bbox_inches='tight')
    plt.close(fig)

# Example usage
plot_inset_cumulative_tp_auc(
    df_listSdhash,  # AutoPYara
    df_listSdhashYARA,  # AutoYara
    drop_smallest_n=0,
    output_file='inset_cumulative_tp2_auc_plot.pdf'
)